# Pothole Detection — YOLOv8 Training (Colab)

Steps in this notebook:
1. Install dependencies
2. Download a pothole dataset (Roboflow) — replace with your own dataset if you have one
3. Train YOLOv8n (nano, best for on-device/mobile)
4. Validate the model
5. Export to TFLite (Android) and CoreML (iOS)
6. Download the exported model files

**Before running:** Go to Runtime > Change runtime type > select GPU (T4 is fine).

In [ ]:
# 1. Install dependencies
!pip install ultralytics roboflow -q

import ultralytics
ultralytics.checks()

## 2. Get a dataset

Option A — use a public Roboflow pothole dataset (quickest way to get started).
Go to https://universe.roboflow.com and search "pothole detection". Pick a dataset,
click **Download**, choose format **YOLOv8**, and copy the generated code snippet
(it includes your own API key) into the cell below, replacing the placeholder.

Option B — upload your own annotated dataset (e.g. zipped in YOLO format) via the
Colab file browser, then skip the roboflow download cell and point `DATA_YAML`
directly at your `data.yaml`.

In [ ]:
# --- Option A: Roboflow dataset download ---
# Replace with the exact snippet Roboflow gives you for your chosen dataset
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
version = project.version(1)  # change version number if needed
dataset = version.download("yolov8")

DATA_YAML = f"{dataset.location}/data.yaml"
print("Dataset ready at:", DATA_YAML)

In [ ]:
# --- Option B: use your own dataset instead ---
# Uncomment and set this if you uploaded your own YOLO-format dataset
# DATA_YAML = "/content/your_dataset/data.yaml"

## 3. Train YOLOv8n

`yolov8n.pt` is the nano variant — smallest and fastest, best suited for on-device
mobile inference after export. Start with 50-100 epochs; adjust based on how
training/validation loss behaves.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # start from pretrained COCO weights

results = model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=640,
    batch=16,
    patience=15,       # early stopping if no improvement
    project="pothole_training",
    name="yolov8n_pothole",
    augment=True,
    device=0            # GPU
)

## 4. Validate

Check precision, recall, and mAP on the validation split. Low recall means the
model is missing real potholes (false negatives) — often fixable with more data
or more epochs. Low precision means too many false positives — check confidence
threshold at inference time.

In [ ]:
best_model_path = "pothole_training/yolov8n_pothole/weights/best.pt"
model = YOLO(best_model_path)

metrics = model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

## 5. Quick test on a sample image

Upload a test road photo to Colab (drag into the file browser on the left) and
run inference to sanity-check detections before exporting.

In [ ]:
# Set this to an uploaded test image path
TEST_IMAGE = "/content/test_road.jpg"

results = model.predict(TEST_IMAGE, conf=0.4, save=True)
for r in results:
    print(f"Detections: {len(r.boxes)}")
    for box in r.boxes:
        print("Confidence:", float(box.conf[0]), "BBox:", box.xyxy[0].tolist())

## 6. Export for mobile

- **TFLite** → for Android (works with `tflite-support` or TensorFlow Lite Interpreter)
- **CoreML** → for iOS

Both exports include the trained weights only — no internet/API calls needed at
inference time, since this runs fully on-device.

In [ ]:
# Export to TFLite (Android)
model.export(format="tflite", imgsz=640, int8=False)  # set int8=True for smaller/faster, needs calibration data

# Export to CoreML (iOS)
model.export(format="coreml", imgsz=640)

In [ ]:
# 7. Download exported models to your machine
from google.colab import files
import glob

tflite_path = glob.glob("pothole_training/yolov8n_pothole/weights/best.tflite")
coreml_path = glob.glob("pothole_training/yolov8n_pothole/weights/best.mlpackage") or \
              glob.glob("pothole_training/yolov8n_pothole/weights/best.mlmodel")

if tflite_path:
    files.download(tflite_path[0])
if coreml_path:
    print("CoreML export at:", coreml_path[0], "— zip the .mlpackage folder before downloading if needed.")

## Next steps

- Drop `best.tflite` into your Android app's assets folder and load it with the
  TensorFlow Lite Interpreter (or `tflite-support` Task Library for simpler
  bounding-box output handling).
- Drop `best.mlpackage` into your iOS app's bundle and load with Vision + CoreML.
- Keep this notebook around — as you collect confirmed reports from real users
  (fixed/still-present confirmations, manual reports), periodically retrain on
  the growing dataset to improve accuracy for your specific region's roads.